In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

import torch
import torch_geometric as pyg

from baseline_evals.feature_selection import variance_filtering

from torch_geometric.utils import to_networkx


from bipartite_gnn.train_test import GNNTrainer
from bipartite_gnn.model import GAT_2L, BipartiteRGAT, BiRGAT
from bipartite_gnn.graph_building import cosine_similarity_matrix, dense_to_coo
from bipartite_gnn.preprocessing import gg_interactions, get_mirna_gene_interactions

In [5]:
# Check currently allocated CUDA memory in bytes
allocated_memory = torch.cuda.memory_allocated()
print(f"Currently allocated CUDA memory: {allocated_memory} bytes")

# Check maximum allocated CUDA memory in bytes
max_allocated_memory = torch.cuda.max_memory_allocated()
print(f"Maximum allocated CUDA memory: {max_allocated_memory} bytes")


Currently allocated CUDA memory: 0 bytes
Maximum allocated CUDA memory: 0 bytes


In [6]:
null_vals = ["NA"]
mrna = pl.read_csv("BRCA_PROCESSED_DATA/mrna.tsv", separator="\t", null_values=null_vals)
cna = pl.read_csv("BRCA_PROCESSED_DATA/cnvth.tsv", separator="\t", null_values=null_vals)
mirna = pl.read_csv("BRCA_PROCESSED_DATA/mirna.tsv", separator="\t", null_values=null_vals)

labels = pl.read_csv("BRCA_PROCESSED_DATA/labels.tsv", separator="\t")
le = LabelEncoder()
le.fit(labels["PAM50_mRNA_nature2012"].to_list())
y = le.transform(labels["PAM50_mRNA_nature2012"].to_list())
# labels, y

In [7]:
# mrna, cna, mirna
mrna_gene_names = mrna[:, 0].to_list()
cna_gene_names = cna[:, 0].to_list()
mirna_gene_names = mirna[:, 0].to_list()

In [8]:
mirna_gene_names = pl.read_csv("miRBaseConverter_Result.csv")
mirna_targets = mirna_gene_names["TargetName"].to_list()

In [9]:
mrna_X = torch.tensor(mrna[:, 1:].to_numpy().T)
cna_X = torch.tensor(cna[:, 1:].to_numpy().T, dtype=torch.float32)
mirna_X = torch.tensor(mirna[:, 1:].to_numpy().T)

In [10]:
train_mask, test_mask = train_test_split(torch.arange(len(y)), test_size=0.4, stratify=y, random_state=5)
val_mask, test_mask = train_test_split(test_mask, test_size=0.5, random_state=5, stratify=y[test_mask])

In [35]:
from bipartite_gnn.graph_building import create_diff_exp_connections_nbnom, create_diff_exp_connections_norm, dense_to_coo


# select 400 top genes for mrna
select_mask = variance_filtering(mrna_X[train_mask].numpy(), 5000)
mrna_X_400 = mrna_X[:, select_mask]

mrna_genes = np.array(mrna_gene_names)[select_mask]
# select 400 top genes for cna
# select_mask = variance_filtering(cna_X.numpy(), 500)
# cna_X_400 = cna_X[:, select_mask].float()
 
mrna_A = create_diff_exp_connections_norm(mrna_X_400, train_mask, 2.5)
# mrna_A_n = create_diff_exp_connections_nbnom(mrna_X_400, train_mask, 2.5)
# mrna_A = torch.logical_or(mrna_A, mrna_A_n).int()

print(mrna_A.shape)

connected_nodes_mask = mrna_A.sum(dim=0) != 0

mrna_A = mrna_A[:, connected_nodes_mask]
mrna_X_400 = mrna_X_400[:, connected_nodes_mask]

mrna_genes = mrna_genes[connected_nodes_mask]

print(mrna_A.shape, mrna_X_400.shape)

print("isolated sample nodes, isolated gene nodes, mean degree: ")
print(
    (mrna_A.abs().sum(axis=1) == 0).sum(),
    (mrna_A.abs().sum(axis=0) == 0).sum(),
    mrna_A.abs().sum() / mrna_A.shape[0],
)

# cna_A = create_diff_exp_connections_norm(cna_X_400, train_mask, 1.0)

mirna_A = create_diff_exp_connections_norm(mirna_X, train_mask, 1.5)

# normalize mrna_X_400
std_scaler = StandardScaler()
std_scaler.fit(mrna_X_400.numpy()[train_mask])

mrna_X_400 = torch.tensor(std_scaler.transform(mrna_X_400.numpy()))

# # # normalize cna_X
# # std_scaler = StandardScaler()
# # std_scaler.fit(cna_X.numpy()[train_mask])

# # cna_X = torch.tensor(std_scaler.transform(cna_X.numpy()))

# normalize mirna_X
std_scaler = StandardScaler()
std_scaler.fit(mirna_X.numpy()[train_mask])

mirna_X = torch.tensor(std_scaler.transform(mirna_X.numpy()))

isolated sample nodes, isolated gene nodes, mean degree: 
tensor(0) tensor(69) tensor(95.2464, dtype=torch.float64)
torch.Size([483, 5000])
torch.Size([483, 4663]) torch.Size([483, 4663])
isolated sample nodes, isolated gene nodes, mean degree: 
tensor(0) tensor(0) tensor(91.0021, dtype=torch.float64)
isolated sample nodes, isolated gene nodes, mean degree: 
tensor(0) tensor(0) tensor(28.9772, dtype=torch.float64)


In [36]:
mrna_features_A = gg_interactions(list(mrna_genes))

1131384it [03:08, 6003.89it/s] 


In [37]:
mirna_mrna_interactions = get_mirna_gene_interactions(mirna_targets, mrna_genes)

torch.Size([231, 4663])


In [41]:
from bipartite_gnn.preprocessing import pp_interactions

gene_gene_A = pp_interactions(mrna_genes, mrna_genes)

In [47]:
mrna_features_A = torch.logical_or(mrna_features_A, gene_gene_A).int()

In [48]:
mirna_mrna_interactions.shape

torch.Size([231, 4663])

In [49]:
# create hetero-data object
import torch_geometric.transforms as T

from bipartite_gnn.graph_building import dense_to_attributes

data = pyg.data.HeteroData()

proj_dim = 200

# sample node features
data["mrna"].x = mrna_X_400.float()
# data["cna"].x = cna_X_400.float()
data["mirna"].x = mirna_X.float()

data.y = torch.tensor(y)

data.omics = ["mrna", "mirna"] #, "cna", "mirna"]
data.feature_names = ["mrna_feature", "mirna_feature"] #, "cna_feature", "mirna_feature"]

# feature node projection
data["mrna_feature"].x = torch.zeros(mrna_X_400.shape[1], proj_dim)
# data["cna_feature"].x = torch.zeros(cna_X_400.shape[1], proj_dim)
data["mirna_feature"].x = torch.zeros(mirna_X.shape[1], proj_dim)

# create edges
 # 3 for forward egdes, 3 for backward edges

data["mrna", "diff_exp", "mrna_feature"].edge_index = dense_to_coo(mrna_A)
data["mrna", "diff_exp", "mrna_feature"].edge_attributes = dense_to_attributes(mrna_A)
data["mrna_feature", "interaction", "mrna_feature"].edge_index = dense_to_coo(mrna_features_A)

data["mirna", "diff_exp", "mirna_feature"].edge_index = dense_to_coo(mirna_A)
data["mirna", "diff_exp", "mirna_feature"].edge_attributes = dense_to_attributes(mirna_A)

data["mirna_feature", "interacts", "mrna_feature"].edge_index = dense_to_coo(mirna_mrna_interactions)

# data["mrna", "self_loop", "mrna"].edge_index = dense_to_coo(torch.eye(mrna_X_400.shape[0]))
# data["mrna_feature", "self_loop", "mrna_feature"].edge_index = dense_to_coo(torch.eye(mrna_X_400.shape[1]))
# data["cna", "diff_exp", "cna_feature"].edge_index = dense_to_coo(cna_A)
# data["mirna", "diff_exp", "mirna_feature"].edge_index = dense_to_coo(mirna_A)

data = T.ToUndirected()(data)
# data = T.AddSelfLoops()(data)

# 2 for forward and backward diff exp, one for each self loop
data.num_relations = len(data.edge_index_dict.keys())
print("num_relations", data.num_relations)

data["train_mask"] = train_mask
data["val_mask"] = val_mask
data["test_mask"] = test_mask

data

num_relations 7


HeteroData(
  y=[483],
  omics=[2],
  feature_names=[2],
  num_relations=7,
  train_mask=[289],
  val_mask=[97],
  test_mask=[97],
  mrna={ x=[483, 4663] },
  mirna={ x=[483, 231] },
  mrna_feature={ x=[4663, 200] },
  mirna_feature={ x=[231, 200] },
  (mrna, diff_exp, mrna_feature)={
    edge_index=[2, 43954],
    edge_attributes=[43954, 1],
  },
  (mrna_feature, interaction, mrna_feature)={ edge_index=[2, 116589] },
  (mirna, diff_exp, mirna_feature)={
    edge_index=[2, 13996],
    edge_attributes=[13996, 1],
  },
  (mirna_feature, interacts, mrna_feature)={ edge_index=[2, 11884] },
  (mrna_feature, rev_diff_exp, mrna)={
    edge_index=[2, 43954],
    edge_attributes=[43954, 1],
  },
  (mirna_feature, rev_diff_exp, mirna)={
    edge_index=[2, 13996],
    edge_attributes=[13996, 1],
  },
  (mrna_feature, rev_interacts, mirna_feature)={ edge_index=[2, 11884] }
)

In [53]:
# model = BipartiteRGAT(
#     input_sizes=[mrna_X_400.shape[1]], # , cna_X_400.shape[1], mirna_X_200.shape[1]],
#     proj_dim=proj_dim,
#     hidden_channels=[128, 64], # num_layers = len(hidden_channels) - 1
#     num_labels=len(np.unique(y)),
#     num_relations=data.num_relations,
#     num_heads=1,
#     feature_integration_mode="linear",
# )

model = BiRGAT(
    omic_channels=data.omics,
    feature_names=data.feature_names,
    relations=list(data.edge_index_dict.keys()),
    input_dims=[mrna_X_400.shape[1], mirna_X.shape[1]], # , cna_X_400.shape[1], mirna_X.shape[1]],
    num_classes=len(np.unique(y)),
    proj_dim=proj_dim,
    hidden_channels=[128, 128, 128, 64],
    heads=3,
    dropout=0.4,
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-3)

ccounts = torch.unique(data.y[train_mask], return_counts=True)[1]
inv_w = 1.0 / ccounts.float()
class_weights = inv_w / inv_w.sum()

trainer = GNNTrainer(
    model=model,
    loss_fn=torch.nn.CrossEntropyLoss(weight=class_weights),
    optimizer=optimizer,
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=15, min_lr=1e-5),
    params={
        "l1_lambda" : 0.01,
    }
)

trainer.train(
    data=data,
    epochs=150,
    log_interval=5,
)

omic_channels:  ['mrna', 'mirna']
feature_names:  ['mrna_feature', 'mirna_feature']
all_names:  ['mrna', 'mirna', 'mrna_feature', 'mirna_feature']
projections:  {'mrna': Linear(4663, 200, bias=True), 'mirna': Linear(231, 200, bias=True)}
Epoch: 005, 
Train Loss: 84.8737, Train Acc: 0.3599, Train F1 Macro: 0.3397, Train F1 Weighted: 0.3409
Val Loss: 1.3590, Val Acc: 0.4124, Val F1 Macro: 0.3463, Val F1 Weighted: 0.3190
Test Loss: 1.3601, Test Acc: 0.4227, Test F1 Macro: 0.3738, Test F1 Weighted: 0.3281
##################################################
Epoch: 010, 
Train Loss: 84.8385, Train Acc: 0.6228, Train F1 Macro: 0.6068, Train F1 Weighted: 0.6308
Val Loss: 1.3268, Val Acc: 0.5979, Val F1 Macro: 0.5012, Val F1 Weighted: 0.5514
Test Loss: 1.3301, Test Acc: 0.6186, Test F1 Macro: 0.5177, Test F1 Weighted: 0.5655
##################################################
Epoch: 015, 
Train Loss: 84.7973, Train Acc: 0.6817, Train F1 Macro: 0.5983, Train F1 Weighted: 0.6508
Val Loss: 1.2797, V

In [ ]:
model(data.clone()).shape

Data(omics=[3], num_relations=6, edge_index=[2, 13842], x=[2449, 128], y=[2449], node_type=[2449], edge_type=[13842])
torch.Size([3, 483, 64])


torch.Size([483])

In [ ]:
dh = data.to_homogeneous()

HeteroData(
  num_relations=6,
  mrna={
    x=[483, 400],
    y=[483],
  },
  cna={
    x=[483, 400],
    y=[483],
  },
  mirna={
    x=[483, 200],
    y=[483],
  },
  mrna_feature={ x=[18975, 128] },
  cna_feature={ x=[24776, 128] },
  mirna_feature={ x=[231, 128] },
  (mrna, diff_exp, mrna_feature)={ edge_index=[2, 3730] },
  (cna, diff_exp, cna_feature)={ edge_index=[2, 1997] },
  (mirna, diff_exp, mirna_feature)={ edge_index=[2, 1194] },
  (mrna_feature, rev_diff_exp, mrna)={ edge_index=[2, 3730] },
  (cna_feature, rev_diff_exp, cna)={ edge_index=[2, 1997] },
  (mirna_feature, rev_diff_exp, mirna)={ edge_index=[2, 1194] }
)

In [ ]:
dh

Data(num_relations=6, edge_index=[2, 13842], x=[45431, 400], y=[45431], node_type=[45431], edge_type=[13842])

In [ ]:
data

HeteroData(
  mrna={
    x=[483, 18975],
    y=[483],
  },
  cna={
    x=[483, 24776],
    y=[483],
  },
  mirna={
    x=[483, 231],
    y=[483],
  },
  mrna_feature={ x=[18975, 128] },
  cna_feature={ x=[24776, 128] },
  mirna_feature={ x=[231, 128] }
)

In [ ]:
mrna_X.shape, cna_X.shape, mirna_X.shape

(torch.Size([483, 18975]), torch.Size([483, 24776]), torch.Size([483, 231]))

In [ ]:
dh

Data(x=[45431, 24776], y=[45431], node_type=[45431])

In [ ]:
# get counts of unique values in the dh.node_type tensor
node_type_counts = torch.unique(dh.node_type, return_counts=True)
node_type_counts

(tensor([0, 1, 2, 3, 4, 5]),
 tensor([  483,   483,   483, 18975, 24776,   231]))

In [ ]:
data

HeteroData(
  mrna={
    x=[18975, 483],
    y=[483],
  },
  cna={
    x=[24776, 483],
    y=[483],
  },
  mirna={
    x=[231, 483],
    y=[483],
  }
)

In [ ]:
# train the model